# Large Neighbourhood Search (LNS) with Attention Model as Repair Operator

## Setup

In [17]:
from pathlib import Path

import torch

from core.lns.acceptance_criterion import GreedyAcceptance
from core.lns.destroy import RandomSegmentDestroy
from core.lns.lns import lns
from core.lns.repair import AttentionModelRepair
from core.models.attention import AttentionModel
from core.models.attention.training import train
from core.models.utils import load_checkpoint, save_checkpoint
from core.problems.tsp import TensorTSPSampler, TSPBatchInstance, TSPConfig
from core.problems.tsp.solve import random_permutation

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NOTEBOOK_PATH = Path(__file__).parent if "__file__" in dir() else Path(".")

print(f"Using device: {DEVICE}")
print(f"Notebook path: {NOTEBOOK_PATH}")

Using device: cuda
Notebook path: .


In [2]:
%load_ext autoreload
%autoreload 2

Train the AM to be used as the repair operator in the LNS.

In [3]:
tsp_sampler = TensorTSPSampler(
    TSPConfig(
        num_locations=20,
        min_coordinate=0.0,
        max_coordinate=1.0,
        distribution_of_locations="uniform",
    ),
)

model = AttentionModel()

model_path = NOTEBOOK_PATH / "checkpoints" / "attention_model_for_repair.pth"
model_path.parent.mkdir(parents=True, exist_ok=True)

if model_path.exists():
    print(f"Loading model from {model_path}")
    load_checkpoint(path=model_path, model=model)
else:
    history = train(
        model=model,
        tsp_sampler=tsp_sampler,
        n_epochs=15,
        steps_per_epoch=100,
        batch_size=512,
        lr=1e-4,
        alpha=0.05,
        device=DEVICE,
    )

    save_checkpoint(
        path=model_path,
        model=model,
    )

Training:   0%|          | 0/15 [00:00<?, ?epoch/s]

Epoch 1:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 2:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 3:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 4:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 5:   0%|          | 0/100 [00:00<?, ?step/s]

... (6 outputs left out)

Epoch 12:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 13:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 14:   0%|          | 0/100 [00:00<?, ?step/s]

Epoch 15:   0%|          | 0/100 [00:00<?, ?step/s]

## Basic LNS

In [45]:
experiments = [
    (20, 20, 1),
    (20, 10, 50),
    (20, 5, 100),
    (50, 15, 50),
    (100, 15, 50),
]

results_of_experiments = []
for num_locations, destroy_size, n_iterations in experiments:
    print(
        f"Experiment: num_locations={num_locations}, destroy_size={destroy_size}, n_iterations={n_iterations}"
    )

    destroy = RandomSegmentDestroy(destroy_size)
    repair = AttentionModelRepair(model)
    accept = GreedyAcceptance()

    sampler = TensorTSPSampler(
        TSPConfig(
            num_locations=num_locations,
            min_coordinate=0.0,
            max_coordinate=1.0,
            distribution_of_locations="uniform",
        ),
        seed=42,
    )
    batch = sampler(512).to(DEVICE)

    initial_solutions = random_permutation(TSPBatchInstance(batch))

    search_result = lns(
        starting_solutions=initial_solutions,
        destroy=destroy,
        repair=repair,
        accept=accept,
        n_iterations=n_iterations,
    )

    results_of_experiments.append(
        {
            "num_locations": num_locations,
            "destroy_size": destroy_size,
            "n_iterations": n_iterations,
            "mean_initial_cost": search_result.initial_lengths.mean().item(),
            "mean_final_cost": search_result.best_lengths.mean().item(),
            "history": search_result.history,
        }
    )

## Training the AM within the LNS loop

In [ ]:
# TODO